### LightGBM (Light Gradient Boosting Machine)

`LightGBM` is an advanced gradient boosting framework designed for speed and efficiency.

- key features:
  - Uses leaf-wise tree growth and histogram-based algorithm for faster training and better accuracy.

  - Handles categorical features natively without one-hot encoding.

  - Supports parallel and GPU learning for scalability and speed.

  - Efficiently manages large datasets with low memory usage and sparse data.

  - Offers advanced optimizations like Gradient-based One-Side Sampling and Exclusive Feature Bundling to enhance performance and reduce overfitting.

  This makes LightGBM highly efficient, scalable, and accurate for large-scale machine learning tasks

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import lightgbm as lgb

In [2]:
# @title
# Load the dataset
df = pd.read_csv('/content/earthquakes_data_preprocessed.csv')

In [3]:
# Prepare features and target variable
X = df.drop(columns='risk_score')
y = df['risk_score']

In [4]:
# Split into train and test sets
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [5]:
# Initialize and train the LightGBM model
model = lgb.LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    random_state=42
)
model.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002446 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1018
[LightGBM] [Info] Number of data points in the train set: 87772, number of used features: 6
[LightGBM] [Info] Start training from score 1.088666
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: 

LGBMRegressor(max_depth=3, random_state=42)

In [6]:
# Make predictions
y_pred = model.predict(X_val)

mae = mean_absolute_error(y_val, y_pred)
mse = mean_squared_error(y_val, y_pred)
r2 = r2_score(y_val, y_pred)
print(f'MAE: {mae:.4f}, MSE: {mse:.4f}, R2: {r2:.4f}')

MAE: 0.0463, MSE: 0.0404, R2: 0.9981


In [7]:
# Hyperparameter tuning using GridSearchCV
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}
grid_search = GridSearchCV(
    estimator=lgb.LGBMRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_mean_absolute_error',
    cv=3,
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.009858 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1018
[LightGBM] [Info] Number of data points in the train set: 87772, number of used features: 6
[LightGBM] [Info] Start training from score 1.088666
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

GridSearchCV(cv=3, estimator=LGBMRegressor(random_state=42), n_jobs=-1,
             param_grid={'learning_rate': [0.05, 0.1], 'max_depth': [3, 5],
                         'n_estimators': [100, 200]},
             scoring='neg_mean_absolute_error')

In [8]:
best_model = grid_search.best_estimator_
y_pred_best = best_model.predict(X_val)
# print('Best MAE:', mean_absolute_error(y_val, y_pred_best))
# print('Best MSE:', mean_squared_error(y_val, y_pred_best))
# print('Best R2:', r2_score(y_val, y_pred_best))
mae = mean_absolute_error(y_val, y_pred_best)
mse = mean_squared_error(y_val, y_pred_best)
r2 = r2_score(y_val, y_pred_best)
print(f'MAE: {mae:.4f}, MSE: {mse:.4f}, R2: {r2:.4f}')

MAE: 0.0221, MSE: 0.0388, R2: 0.9982


#Observation

Histogram-based algorithms → lightning fast

Leaf-wise growth instead of level-wise → deeper trees, smarter splits

GPU acceleration

Huge datasets (millions of rows) handled effortlessly